# Notebook 17 — Cross-Dataset Transformer Validation of VoxIntel-R

## Why this notebook exists

Notebook 16 introduced the first reference-free VoxIntel-R experiment on SLURP. It tested
whether downstream intent failure can be predicted using only signals available at
inference time — ASR-native uncertainty and intent-model uncertainty — without access to
the reference transcript.

That experiment produced a useful but unresolved result: intent-model uncertainty was
strongly predictive of downstream failure, while adding ASR-native uncertainty features
did not clearly improve on the intent-only baseline. That result was obtained within a
single dataset and a single model family, and the Notebook 16 evaluation population still
needs a final fixed-split rerun before its numbers are treated as canonical.

This notebook asks a different question:

> Does the reference-free VoxIntel-R reliability signal generalize to a different
> spoken-language-understanding dataset and independently trained Transformer models?

We use **Fluent Speech Commands (FSC)** as an external validation dataset — 30,043 spoken
commands from 97 speakers across 31 intents, with official speaker-independent
train/validation/test splits.

The goal is **not** to search for a dataset on which H1 happens to succeed. The goal is to
test whether the same reference-free risk formulation survives a change in:

- dataset
- speakers
- command vocabulary
- acoustic conditions
- ASR model checkpoint
- intent Transformer checkpoint

The experiment reproduces the core VoxIntel-R protocol from Notebook 16 while keeping the
feature definitions and the leakage boundary fixed.

### Research hypothesis

**H1 — Cross-dataset complementarity**

> Reference-free ASR-native and intent-native uncertainty signals contain complementary
> information for predicting downstream intent failure, such that their combination
> improves risk prediction over intent uncertainty alone on an unseen spoken-command
> dataset.

### Critical anti-leakage rule

The reference transcript may be used to construct the supervised target and for final
evaluation, but it must **never** be used to construct an inference-time risk feature.

**Allowed:** ASR logits/probabilities · ASR hypothesis · audio duration · intent
logits/probabilities

**Forbidden:** reference transcript · WER/CER · reference/hypothesis alignment · lexical
overlap with the reference · reference-derived error taxonomy

The experiment is considered invalid if any forbidden signal enters the risk feature
matrix. Section 15 audits this automatically.

### Sequencing note

This notebook is inserted **between** Notebook 16 and the calibration notebook. The
finalized README originally scheduled calibration + selective prediction as Notebook 17;
this insertion is a deliberate research-sequencing decision (Notebook 16's SLURP result is
provisional and H1 is not yet supported there), not an accidental drift. The README's
roadmap section is updated separately, after this notebook produces an actual result —
not before.

    16  Reference-free VoxIntel-R on SLURP
            ↓
    17  Cross-dataset Transformer validation   ← this notebook
            ↓
    18  Calibration + selective prediction
            ↓
    19  Cost-sensitive selective prediction

## 01 — Research contract

Fixes the notebook's identity, dataset, hypothesis, and seed before any code that could
be influenced by them runs.

In [65]:
NOTEBOOK_ID = "17"
DATASET = "FSC"                 # Fluent Speech Commands — external validation dataset
HYPOTHESIS = "H1"                # Cross-dataset complementarity (see intro cell)
RANDOM_SEED = 42

import random
import numpy as np
import torch

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("=" * 50)
print(f"Notebook:        {NOTEBOOK_ID}")
print(f"Dataset:         Fluent Speech Commands")
print(f"Purpose:         external validation of VoxIntel-R (Notebook 16 protocol)")
print(f"Hypothesis:      {HYPOTHESIS} — cross-dataset complementarity")
print(f"Reference-free:  YES (see anti-leakage audit, Section 15)")
print(f"Random seed:     {RANDOM_SEED}")
print("=" * 50)

Notebook:        17
Dataset:         Fluent Speech Commands
Purpose:         external validation of VoxIntel-R (Notebook 16 protocol)
Hypothesis:      H1 — cross-dataset complementarity
Reference-free:  YES (see anti-leakage audit, Section 15)
Random seed:     42


## 02 — Imports

Reuses the existing project stack wherever the existing `src` modules are already generic
(the audio loader). Everything specific to reference-aware taxonomy machinery from
Notebooks 09–15 is deliberately **not** imported — this notebook is self-contained and
reference-free by construction.

In [66]:
from pathlib import Path
import sys
import json
import math
import random
import warnings

# ---------------------------------------------------------------------------
# PROJECT ROOT / IMPORT PATH
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Users\ACER\OneDrive\Desktop\VoxIntel").resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"VoxIntel project root is invalid.\n"
        f"PROJECT_ROOT: {PROJECT_ROOT}\n"
        f"Expected: {PROJECT_ROOT / 'src'}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"src exists:   {(PROJECT_ROOT / 'src').is_dir()}")

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
import torchaudio

from transformers import (
    AutoProcessor,
    Wav2Vec2ForCTC,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from src.utils.audio import load_audio

print("Imports OK.")

Project root: C:\Users\ACER\OneDrive\Desktop\VoxIntel
src exists:   True
Imports OK.


## 03 — Configuration

Locks every path up front. FSC's official release layout is:

    data/
        train_data.csv
        valid_data.csv
        test_data.csv
    wavs/
        speakers/

which is kept separate from the SLURP paths used elsewhere in the project.

In [67]:
# --- FSC dataset paths ------------------------------------------------------
FSC_ROOT = PROJECT_ROOT / "data" / "raw" / "fsc" / "fluent_speech_commands_dataset"
FSC_DATA_DIR = FSC_ROOT / "data"
FSC_AUDIO_DIR = FSC_ROOT / "wavs"

FSC_TRAIN_CSV = FSC_DATA_DIR / "train_data.csv"
FSC_VALID_CSV = FSC_DATA_DIR / "valid_data.csv"
FSC_TEST_CSV = FSC_DATA_DIR / "test_data.csv"

# --- Model output paths ------------------------------------------------------
MODEL_ROOT = PROJECT_ROOT / "models"
FSC_ASR_MODEL_DIR = MODEL_ROOT / "wav2vec2_fsc"
FSC_INTENT_MODEL_DIR = MODEL_ROOT / "distilbert_fsc_intent"

# Base checkpoints — same model families as the original SLURP pipeline
# (Notebooks 03-06 / 07), so the cross-dataset comparison stays interpretable.
# Explicitly NOT the SLURP fine-tuned checkpoints: FSC needs its own.
BASE_ASR_MODEL = "facebook/wav2vec2-base-960h"
BASE_INTENT_MODEL = "distilbert-base-uncased"

# --- Report / artifact paths -------------------------------------------------
REPORT_DIR = PROJECT_ROOT / "reports"

FSC_AUDIT_PATH = REPORT_DIR / "fsc_dataset_audit.csv"
FSC_ASR_PRED_PATH = REPORT_DIR / "fsc_asr_predictions.csv"
FSC_INTENT_PRED_PATH = REPORT_DIR / "fsc_intent_predictions.csv"
FEATURE_PATH = REPORT_DIR / "fsc_voxintel_r_features.csv"
RESULTS_PATH = REPORT_DIR / "fsc_voxintel_r_model_comparison.csv"
PREDICTIONS_PATH = REPORT_DIR / "fsc_voxintel_r_predictions.csv"
IMPORTANCE_PATH = REPORT_DIR / "fsc_voxintel_r_feature_importance.csv"
CROSS_DATASET_PATH = REPORT_DIR / "fsc_voxintel_r_cross_dataset_comparison.csv"
SUMMARY_PATH = REPORT_DIR / "fsc_voxintel_r_summary.json"

for p in [MODEL_ROOT, REPORT_DIR, FSC_ASR_MODEL_DIR, FSC_INTENT_MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# --- The exact Notebook 16 reference-free feature schema (must match) -------
ASR_FEATURES = [
    "asr_mean_confidence",
    "asr_min_confidence",
    "asr_std_confidence",
    "asr_median_confidence",
    "asr_mean_entropy",
    "asr_max_entropy",
    "asr_std_entropy",
    "asr_num_frames",
    "audio_duration",
]

INTENT_FEATURES = [
    "intent_confidence",
    "intent_entropy",
    "intent_margin",
]

FORBIDDEN_FEATURES = {
    "ground_truth_intent",
    "reference_transcript",
    "wer",
    "cer",
    "taxonomy",
    "error_type",
    "lexical_overlap",
}

print("Configuration locked.")
print(f"FSC root:          {FSC_ROOT}")
print(f"ASR feature count:    {len(ASR_FEATURES)}")
print(f"Intent feature count: {len(INTENT_FEATURES)}")

Configuration locked.
FSC root:          C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset
ASR feature count:    9
Intent feature count: 3


## 04 — Dataset download / structure verification

FSC is ~30k audio files — this notebook does **not** download it automatically. It only
verifies the expected structure and fails loudly with instructions if the data is
missing.

Official dataset page:
[Fluent Speech Commands](https://fluent.ai/fluent-speech-commands-a-dataset-for-spoken-language-understanding-research/)

Download and extract under `data/raw/fsc/` before running the cell below.

In [68]:
FSC_SETUP_INSTRUCTIONS = f"""
FSC dataset not found.

Download:
  https://fluent.ai/fluent-speech-commands-a-dataset-for-spoken-language-understanding-research/

Expected structure after extraction:
  {FSC_ROOT}/
      data/
          train_data.csv
          valid_data.csv
          test_data.csv
      wavs/
          speakers/
"""

missing = [
    p
    for p in [
        FSC_ROOT,
        FSC_TRAIN_CSV,
        FSC_VALID_CSV,
        FSC_TEST_CSV,
        FSC_AUDIO_DIR,
    ]
    if not p.exists()
]

if missing:
    print(FSC_SETUP_INSTRUCTIONS)
    print("Missing paths:")
    for p in missing:
        print(f"  - {p}")

    raise FileNotFoundError(
        "FSC dataset is not present at the expected location."
    )

print("FSC dataset structure verified.")
print(f"  train: {FSC_TRAIN_CSV}")
print(f"  valid: {FSC_VALID_CSV}")
print(f"  test:  {FSC_TEST_CSV}")
print(f"  audio: {FSC_AUDIO_DIR}")

FSC dataset structure verified.
  train: C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset\data\train_data.csv
  valid: C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset\data\valid_data.csv
  test:  C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset\data\test_data.csv
  audio: C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\fsc\fluent_speech_commands_dataset\wavs


## 05 — FSC dataset audit

Reports split sizes against the official published numbers and runs integrity checks
before any modeling happens — missing files, duplicates, and (critically) **speaker
overlap across splits**, since FSC's value here depends on its splits actually being
speaker-independent rather than just documented as such.

In [ ]:
train_df = pd.read_csv(FSC_TRAIN_CSV)
valid_df = pd.read_csv(FSC_VALID_CSV)
test_df = pd.read_csv(FSC_TEST_CSV)

def normalize_fsc_audio_path(path):
    path = Path(str(path))

    # CSV stores paths relative to FSC_ROOT:
    # wavs/speakers/<speaker>/<file>.wav
    # But FSC_AUDIO_DIR already points to FSC_ROOT/wavs.
    if path.parts and path.parts[0].lower() == "wavs":
        path = Path(*path.parts[1:])

    return str(path)


for df in [train_df, valid_df, test_df]:
    df["path"] = df["path"].map(normalize_fsc_audio_path)

EXPECTED_COUNTS = {"train": 23132, "validation": 3118, "test": 3793}
actual_counts = {"train": len(train_df), "validation": len(valid_df), "test": len(test_df)}

print("Split sizes (actual vs. officially published):")
for split, expected in EXPECTED_COUNTS.items():
    actual = actual_counts[split]
    flag = "OK" if actual == expected else "MISMATCH — proceeding with actual counts"
    print(f"  {split:<12} actual={actual:<7} expected={expected:<7} [{flag}]")

# Integrity checks -------------------------------------------------------
audit_rows = []
for name, df in [("train", train_df), ("validation", valid_df), ("test", test_df)]:
    n_missing_audio = df["path"].apply(
        lambda p: not (FSC_ROOT / p).exists()
    ).sum() if "path" in df.columns else None
    n_dup_ids = df.duplicated(subset=[df.columns[0]]).sum()
    n_dup_transcripts = df["transcription"].duplicated().sum() if "transcription" in df.columns else None
    audit_rows.append({
        "split": name,
        "n_rows": len(df),
        "n_missing_audio": n_missing_audio,
        "n_duplicate_ids": n_dup_ids,
        "n_duplicate_transcripts": n_dup_transcripts,
        "n_speakers": df["speakerId"].nunique() if "speakerId" in df.columns else None,
        "n_intents": (df["action"] + "|" + df["object"] + "|" + df["location"]).nunique()
            if set(["action", "object", "location"]).issubset(df.columns) else None,
    })

audit_df = pd.DataFrame(audit_rows)
print()
print(audit_df.to_string(index=False))

# Critical check: splits must be speaker-disjoint -------------------------
if "speakerId" in train_df.columns:
    train_speakers = set(train_df["speakerId"])
    valid_speakers = set(valid_df["speakerId"])
    test_speakers = set(test_df["speakerId"])

    assert train_speakers.isdisjoint(valid_speakers), "Speaker leakage: train/validation overlap"
    assert train_speakers.isdisjoint(test_speakers), "Speaker leakage: train/test overlap"
    assert valid_speakers.isdisjoint(test_speakers), "Speaker leakage: validation/test overlap"

    print()
    print(f"Speaker-independence check PASSED "
          f"(train={len(train_speakers)}, valid={len(valid_speakers)}, test={len(test_speakers)} speakers)")

audit_df.to_csv(FSC_AUDIT_PATH, index=False)
print(f"\nSaved: {FSC_AUDIT_PATH}")

Split sizes (actual vs. officially published):
  train        actual=23132   expected=23132   [OK]
  validation   actual=3118    expected=3118    [OK]
  test         actual=3793    expected=3793    [OK]

     split  n_rows  n_missing_audio  n_duplicate_ids  n_duplicate_transcripts  n_speakers  n_intents
     train   23132                0                0                    22884          77         31
validation    3118                0                0                     2870          10         31
      test    3793                0                0                     3545          10         31

Speaker-independence check PASSED (train=77, valid=10, test=10 speakers)

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_dataset_audit.csv


## 06 — Label mapping

Intent labels are built **from training data only** — never from validation or test — to
avoid quietly leaking label-space information from held-out splits. FSC's `action` /
`object` / `location` columns are combined into a single intent label, matching the
single-label classification setup used for SLURP intent in Notebook 07.

In [70]:
def fsc_intent_label(df: pd.DataFrame) -> pd.Series:
    return df["action"].astype(str) + "|" + df["object"].astype(str) + "|" + df["location"].astype(str)

train_df["intent_label"] = fsc_intent_label(train_df)
valid_df["intent_label"] = fsc_intent_label(valid_df)
test_df["intent_label"] = fsc_intent_label(test_df)

# Label space is derived from TRAIN ONLY.
unique_train_intents = sorted(train_df["intent_label"].unique())
intent_to_id = {intent: i for i, intent in enumerate(unique_train_intents)}
id_to_intent = {i: intent for intent, i in intent_to_id.items()}

n_labels = len(intent_to_id)
print(f"Number of labels (from training data): {n_labels}")
assert n_labels == 31, f"Expected 31 FSC intents, found {n_labels}"

# Any validation/test intent not seen in training is dropped from modeling
# and reported explicitly rather than silently coerced.
for name, df in [("validation", valid_df), ("test", test_df)]:
    unseen = set(df["intent_label"]) - set(intent_to_id)
    if unseen:
        print(f"WARNING: {len(unseen)} unseen intent label(s) in {name}, dropping those rows: {unseen}")
        df.drop(df[df["intent_label"].isin(unseen)].index, inplace=True)

train_df["intent_id"] = train_df["intent_label"].map(intent_to_id)
valid_df["intent_id"] = valid_df["intent_label"].map(intent_to_id)
test_df["intent_id"] = test_df["intent_label"].map(intent_to_id)

print("Label mapping built from training data only.")

Number of labels (from training data): 31
Label mapping built from training data only.


## 07 — Train FSC ASR Transformer

Fine-tunes a **fresh** `wav2vec2-base-960h` checkpoint on FSC training audio. This is a
new checkpoint — the SLURP fine-tuned ASR model is not reused, since the whole point of
this notebook is an independently trained model on an independent dataset.

    FSC train
        ↓
    Wav2Vec2ForCTC
        ↓
    FSC ASR checkpoint  →  models/wav2vec2_fsc/

Training loop is intentionally left as a call to a `finetune_wav2vec2_ctc(...)` helper
rather than inlined boilerplate — swap in the project's standard CTC fine-tuning loop
here. Character-level CTC on the FSC transcription field is standard for this dataset.

In [71]:
processor = AutoProcessor.from_pretrained(BASE_ASR_MODEL)
asr_model = Wav2Vec2ForCTC.from_pretrained(BASE_ASR_MODEL)


def finetune_wav2vec2_ctc(
    model,
    processor,
    train_df,
    valid_df,
    audio_root: Path,
    output_dir: Path,
    num_epochs: int = 10,
    batch_size: int = 8,
):
    """
    Fine-tune Wav2Vec2ForCTC on FSC using standard CTC fine-tuning.

    Left as a thin wrapper: plug in the project's existing CTC training loop
    (data collator with dynamic padding, AdamW, linear warmup) here. Kept
    local to this notebook rather than pulled from `src/asr`, since that
    module's SLURP-specific training path hasn't been shown to be generic.
    """
    raise NotImplementedError(
        "Wire this up to the project's CTC training loop before running "
        "end-to-end. Left explicit rather than faked so the notebook "
        "doesn't silently skip training."
    )


# finetune_wav2vec2_ctc(
#     asr_model,
#     processor,
#     train_df,
#     valid_df,
#     audio_root=FSC_AUDIO_DIR,
#     output_dir=FSC_ASR_MODEL_DIR,
# )

# processor.save_pretrained(FSC_ASR_MODEL_DIR)

print(f"FSC ASR checkpoint target: {FSC_ASR_MODEL_DIR}")

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 308.00it/s, Materializing param=wav2vec2.feature_projection.projection.weight]                         
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FSC ASR checkpoint target: C:\Users\ACER\OneDrive\Desktop\VoxIntel\models\wav2vec2_fsc


## 08 — FSC ASR evaluation

Establishes the upstream ASR baseline (WER/CER on validation and test). This is reported
for context only — **WER/CER never enter the risk feature matrix** (Section 15 enforces
this).

In [72]:
def compute_wer_cer(predictions: list[str], references: list[str]) -> dict:
    """Word/character error rate. Swap in `jiwer` if available in the environment."""
    try:
        import jiwer
        return {
            "wer": jiwer.wer(references, predictions),
            "cer": jiwer.cer(references, predictions),
        }
    except ImportError:
        raise ImportError(
            "Install `jiwer` to compute WER/CER (pip install jiwer)."
        )


# fsc_asr_model = Wav2Vec2ForCTC.from_pretrained(FSC_ASR_MODEL_DIR)
# fsc_processor = AutoProcessor.from_pretrained(FSC_ASR_MODEL_DIR)
#
# valid_wer_cer = compute_wer_cer(
#     valid_predictions,
#     valid_df["transcription"].tolist()
# )
#
# test_wer_cer = compute_wer_cer(
#     test_predictions,
#     test_df["transcription"].tolist()
# )
#
# print(f"FSC ASR — validation WER/CER: {valid_wer_cer}")
# print(f"FSC ASR — test WER/CER:       {test_wer_cer}")

print("ASR evaluation is an upstream sanity check only — not a risk-model input.")

ASR evaluation is an upstream sanity check only — not a risk-model input.


## 09 — Train FSC intent Transformer

Fine-tunes a fresh `distilbert-base-uncased` as a 31-class intent classifier on FSC
**reference transcripts** — same architecture family as Notebook 07's SLURP intent model,
new checkpoint, new dataset.

    FSC reference transcripts
            ↓
        DistilBERT
            ↓
    31-class intent classifier  →  models/distilbert_fsc_intent/

In [73]:
tokenizer = AutoTokenizer.from_pretrained(BASE_INTENT_MODEL)

intent_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_INTENT_MODEL,
    num_labels=n_labels,
)


def tokenize_intent_batch(texts: list[str]):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )


def finetune_intent_classifier(
    model,
    tokenizer,
    train_df,
    valid_df,
    output_dir: Path,
    num_epochs: int = 5,
    batch_size: int = 16,
):
    """
    Fine-tune DistilBERT for FSC intent classification on ground-truth
    transcripts (`transcription` column), analogous to Notebook 07.
    """
    raise NotImplementedError(
        "Wire this up to a standard HF Trainer classification loop before "
        "running end-to-end. Left explicit rather than faked so the notebook "
        "doesn't silently skip training."
    )


# finetune_intent_classifier(
#     intent_model,
#     tokenizer,
#     train_df,
#     valid_df,
#     output_dir=FSC_INTENT_MODEL_DIR,
# )

# intent_model.save_pretrained(FSC_INTENT_MODEL_DIR)
# tokenizer.save_pretrained(FSC_INTENT_MODEL_DIR)

print(f"FSC intent checkpoint target: {FSC_INTENT_MODEL_DIR}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 369.70it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FSC intent checkpoint target: C:\Users\ACER\OneDrive\Desktop\VoxIntel\models\distilbert_fsc_intent


## 10 — Clean-text intent upper bound

The FSC analogue of Notebook 07: intent performance on the **ground-truth transcript**,
establishing the ceiling before ASR errors are introduced.

In [74]:
def top_k_accuracy(logits: torch.Tensor, labels: torch.Tensor, k: int = 3) -> float:
    topk = logits.topk(k, dim=-1).indices
    correct = (topk == labels.unsqueeze(-1)).any(dim=-1)
    return correct.float().mean().item()

# fsc_intent_model = AutoModelForSequenceClassification.from_pretrained(FSC_INTENT_MODEL_DIR)
# fsc_tokenizer = AutoTokenizer.from_pretrained(FSC_INTENT_MODEL_DIR)
#
# clean_logits, clean_labels = run_intent_inference(
#     fsc_intent_model, fsc_tokenizer, test_df["transcription"], test_df["intent_id"]
# )
# clean_preds = clean_logits.argmax(dim=-1)
#
# clean_text_metrics = {
#     "accuracy": accuracy_score(clean_labels, clean_preds),
#     "macro_f1": f1_score(clean_labels, clean_preds, average="macro"),
#     "weighted_f1": f1_score(clean_labels, clean_preds, average="weighted"),
#     "top3_accuracy": top_k_accuracy(clean_logits, torch.tensor(clean_labels), k=3),
# }
# print("Clean-text intent upper bound (FSC test):")
# for k, v in clean_text_metrics.items():
#     print(f"  {k}: {v:.4f}")

print("Clean-text upper bound establishes the ceiling for the ASR-propagated numbers in Section 11.")

Clean-text upper bound establishes the ceiling for the ASR-propagated numbers in Section 11.


## 11 — ASR → intent propagation

Reproduces Notebook 08's logic on FSC: run the FSC ASR hypothesis through the FSC intent
model, and compare against the clean-transcript ceiling from Section 10.

    audio
     ↓
    FSC Wav2Vec2  →  ASR hypothesis
     ↓
    FSC DistilBERT  →  intent prediction

In [75]:
def transcribe_dataset(
    asr_model,
    processor,
    df: pd.DataFrame,
    audio_root: Path,
) -> pd.DataFrame:
    """
    Runs ASR inference over every utterance, keeping per-utterance
    frame-level logits available for uncertainty extraction in Section 13.
    """
    records = []

    for _, row in df.iterrows():
        audio = load_audio(audio_root / row["path"])

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
        )

        with torch.no_grad():
            logits = asr_model(**inputs).logits  # (1, T, vocab)

        pred_ids = logits.argmax(dim=-1)
        hypothesis = processor.batch_decode(pred_ids)[0]

        records.append({
            "utterance_id": row.get("path", None),
            "asr_hypothesis": hypothesis,
            "audio_duration": audio.shape[-1] / 16000,
            "_logits": logits.squeeze(0),
        })

    return pd.DataFrame(records)


# ------------------------------------------------------------
# ASR → intent propagation
# ------------------------------------------------------------

# asr_results = transcribe_dataset(
#     fsc_asr_model,
#     fsc_processor,
#     test_df,
#     FSC_AUDIO_DIR,
# )

# asr_propagated_logits, asr_propagated_labels = run_intent_inference(
#     fsc_intent_model,
#     fsc_tokenizer,
#     asr_results["asr_hypothesis"],
#     test_df["intent_id"],
# )

# asr_propagated_preds = asr_propagated_logits.argmax(dim=-1)

# propagated_metrics = {
#     "accuracy": accuracy_score(
#         asr_propagated_labels,
#         asr_propagated_preds,
#     ),
#     "macro_f1": f1_score(
#         asr_propagated_labels,
#         asr_propagated_preds,
#         average="macro",
#     ),
# }

# print("Ground truth vs. ASR-propagated intent accuracy on FSC test:")
# print(f"  clean:          {clean_text_metrics['accuracy']:.4f}")
# print(f"  ASR-propagated: {propagated_metrics['accuracy']:.4f}")
# print(
#     f"  recovery gap:   "
#     f"{clean_text_metrics['accuracy'] - propagated_metrics['accuracy']:.4f}"
# )

asr_results = None

print(
    "Propagation cell defined; requires trained FSC ASR + "
    "intent checkpoints to run."
)

Propagation cell defined; requires trained FSC ASR + intent checkpoints to run.


## 12 — Construct the failure target

The supervised target, exactly as defined for SLURP in Notebook 08. The ground-truth
intent is allowed to construct the **target** — it is never allowed into the inference-time
feature matrix (enforced in Section 15).

In [76]:
def build_failure_target(df: pd.DataFrame, predicted_col: str, ground_truth_col: str) -> pd.DataFrame:
    df = df.copy()
    df["intent_failed"] = (df[predicted_col] != df[ground_truth_col]).astype(int)
    return df

# risk_df = build_failure_target(
#     result_df, predicted_col="predicted_intent", ground_truth_col="ground_truth_intent"
# )
#
# failure_rate = risk_df["intent_failed"].mean()
# print(f"Failure prevalence: {failure_rate:.4f}")
# print(f"Success prevalence: {1 - failure_rate:.4f}")

print("Target definition fixed: intent_failed = (predicted_intent != ground_truth_intent).")

Target definition fixed: intent_failed = (predicted_intent != ground_truth_intent).


## 13 — Extract ASR-native uncertainty

Same feature schema as Notebook 16, computed from CTC frame-level logits. Naming note
carried over from the SLURP notebook: these are **frame-level CTC uncertainty
statistics**, not true decoded-token confidence — the notebook names them accordingly
rather than overclaiming token-level confidence.

In [77]:
def extract_asr_uncertainty(
    logits: torch.Tensor,
    audio_duration: float,
) -> dict:
    """
    Frame-level CTC uncertainty statistics.

    `logits`: (T, vocab) raw CTC logits for one utterance.
    """
    probs = F.softmax(logits, dim=-1)

    frame_confidence = probs.max(dim=-1).values

    frame_entropy = -(
        probs * probs.clamp_min(1e-12).log()
    ).sum(dim=-1)

    return {
        "asr_mean_confidence": frame_confidence.mean().item(),
        "asr_min_confidence": frame_confidence.min().item(),
        "asr_std_confidence": frame_confidence.std(
            unbiased=False
        ).item(),
        "asr_median_confidence": frame_confidence.median().item(),
        "asr_mean_entropy": frame_entropy.mean().item(),
        "asr_max_entropy": frame_entropy.max().item(),
        "asr_std_entropy": frame_entropy.std(
            unbiased=False
        ).item(),
        "asr_num_frames": logits.shape[0],
        "audio_duration": audio_duration,
    }


# asr_feature_rows = [
#     extract_asr_uncertainty(
#         row["_logits"],
#         row["audio_duration"],
#     )
#     for _, row in asr_results.iterrows()
# ]

# asr_feature_df = pd.DataFrame(asr_feature_rows)

# assert list(asr_feature_df.columns) == ASR_FEATURES

print(
    f"ASR-native uncertainty schema "
    f"({len(ASR_FEATURES)} features): {ASR_FEATURES}"
)

ASR-native uncertainty schema (9 features): ['asr_mean_confidence', 'asr_min_confidence', 'asr_std_confidence', 'asr_median_confidence', 'asr_mean_entropy', 'asr_max_entropy', 'asr_std_entropy', 'asr_num_frames', 'audio_duration']


## 14 — Extract intent-native uncertainty

Same three features as Notebook 16, computed from the intent classifier's softmax output
over the ASR-propagated hypothesis.

In [78]:
def extract_intent_uncertainty(logits: torch.Tensor) -> dict:
    """`logits`: (num_labels,) raw classification logits for one utterance."""
    probs = F.softmax(logits, dim=-1)
    sorted_probs, _ = probs.sort(descending=True)

    top1 = sorted_probs[0].item()
    top2 = sorted_probs[1].item()

    entropy = -(
        probs * probs.clamp_min(1e-12).log()
    ).sum().item()

    return {
        "intent_confidence": top1,
        "intent_margin": top1 - top2,
        "intent_entropy": entropy,
    }


# intent_feature_rows = [
#     extract_intent_uncertainty(row)
#     for row in asr_propagated_logits
# ]

# intent_feature_df = pd.DataFrame(intent_feature_rows)

# assert list(intent_feature_df.columns) == INTENT_FEATURES

print(
    f"Intent-native uncertainty schema "
    f"({len(INTENT_FEATURES)} features): {INTENT_FEATURES}"
)

Intent-native uncertainty schema (3 features): ['intent_confidence', 'intent_entropy', 'intent_margin']


## 15 — Hard leakage audit

The single most important cell in this notebook. Fails loudly — rather than silently
producing an invalid experiment — if any forbidden signal has made its way into the
feature matrix.

In [79]:
def leakage_audit(feature_columns: list[str]) -> None:
    overlap = set(feature_columns) & FORBIDDEN_FEATURES
    assert not overlap, f"LEAKAGE DETECTED — forbidden features present: {overlap}"

# feature_columns = list(asr_feature_df.columns) + list(intent_feature_df.columns)
# leakage_audit(feature_columns)
#
# print("REFERENCE-FREE AUDIT")
# print("-" * 20)
# print(f"Reference transcript in X: {'YES' if 'reference_transcript' in feature_columns else 'NO'}")
# print(f"Ground-truth intent in X:  {'YES' if 'ground_truth_intent' in feature_columns else 'NO'}")
# print(f"WER/CER in X:              {'YES' if ({'wer','cer'} & set(feature_columns)) else 'NO'}")
# print(f"Taxonomy in X:             {'YES' if 'taxonomy' in feature_columns else 'NO'}")
# print(f"ASR hypothesis available:  YES")
# print(f"ASR uncertainty available: YES")
# print(f"Intent uncertainty available: YES")
# print()
# print("STATUS: PASS")

print(f"Forbidden feature set: {sorted(FORBIDDEN_FEATURES)}")
print("Audit function defined — run after feature assembly, before any modeling.")

Forbidden feature set: ['cer', 'error_type', 'ground_truth_intent', 'lexical_overlap', 'reference_transcript', 'taxonomy', 'wer']
Audit function defined — run after feature assembly, before any modeling.


## 16 — A/B/C risk experiments

Same three experiment arms and the same two model families as Notebook 16 —
`LogisticRegression` and `RandomForestClassifier`. No new model families (no XGBoost,
LightGBM, or neural MLPs): the goal here is dataset/model-family validation, not a model
zoo.

- **A** — ASR-native uncertainty only
- **B** — Intent-native uncertainty only
- **C** — ASR + intent uncertainty combined

In [80]:
RISK_MODELS = {
    "logistic_regression": lambda: LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    "random_forest": lambda: RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED),
}

EXPERIMENTS = {
    "A_asr_only": ASR_FEATURES,
    "B_intent_only": INTENT_FEATURES,
    "C_combined": ASR_FEATURES + INTENT_FEATURES,
}

def evaluate_risk_model(model, X_train, y_train, X_test, y_test) -> dict:
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return {
        "roc_auc": roc_auc_score(y_test, probs),
        "pr_auc": average_precision_score(y_test, probs),
        "brier": brier_score_loss(y_test, probs),
        "f1": f1_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
    }

def run_ab_c_experiments(feature_df: pd.DataFrame, target: pd.Series,
                          train_idx, test_idx) -> pd.DataFrame:
    rows = []
    for exp_name, feature_cols in EXPERIMENTS.items():
        X_train, X_test = feature_df.loc[train_idx, feature_cols], feature_df.loc[test_idx, feature_cols]
        y_train, y_test = target.loc[train_idx], target.loc[test_idx]
        for model_name, model_fn in RISK_MODELS.items():
            metrics = evaluate_risk_model(model_fn(), X_train, y_train, X_test, y_test)
            rows.append({"experiment": exp_name, "model": model_name, **metrics})
    return pd.DataFrame(rows)

# risk_results_df = run_ab_c_experiments(
#     full_feature_df, risk_df["intent_failed"], train_idx=valid_index, test_idx=test_index
# )
# risk_results_df.to_csv(RESULTS_PATH, index=False)
# print(risk_results_df.to_string(index=False))

print("A/B/C harness defined: fit on FSC validation, evaluate on FSC test (see Section 30 split policy).")

A/B/C harness defined: fit on FSC validation, evaluate on FSC test (see Section 30 split policy).


## 17 — Statistical comparison

The primary comparison is **C vs. B** — does ASR-native uncertainty add anything beyond
intent-native uncertainty alone? Not "does C beat A" (it almost certainly will).

Paired bootstrap confidence intervals on the ROC-AUC / PR-AUC / Brier deltas, matching
the methodology used for the Notebook 15 combined-vs-proxy comparison.

In [81]:
def paired_bootstrap_delta(
    y_true,
    probs_a,
    probs_b,
    metric_fn,
    n_boot: int = 2000,
    seed: int = RANDOM_SEED,
) -> dict:
    """
    Bootstrap CI for:
        metric_fn(y_true, probs_b) - metric_fn(y_true, probs_a)

    Resamples utterances jointly so the paired structure is preserved.
    """
    rng = np.random.default_rng(seed)

    y_true = np.asarray(y_true)
    probs_a = np.asarray(probs_a)
    probs_b = np.asarray(probs_b)

    n = len(y_true)

    deltas = np.empty(n_boot)

    for i in range(n_boot):
        idx = rng.integers(0, n, n)

        deltas[i] = (
            metric_fn(y_true[idx], probs_b[idx])
            - metric_fn(y_true[idx], probs_a[idx])
        )

    point_estimate = (
        metric_fn(y_true, probs_b)
        - metric_fn(y_true, probs_a)
    )

    ci_lower, ci_upper = np.percentile(
        deltas,
        [2.5, 97.5],
    )

    return {
        "delta": point_estimate,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "significant": bool(
            ci_lower > 0 or ci_upper < 0
        ),
    }


# c_vs_b = {
#     "roc_auc": paired_bootstrap_delta(
#         y_test,
#         probs_B,
#         probs_C,
#         roc_auc_score,
#     ),
#
#     "pr_auc": paired_bootstrap_delta(
#         y_test,
#         probs_B,
#         probs_C,
#         average_precision_score,
#     ),
#
#     "brier": paired_bootstrap_delta(
#         y_test,
#         probs_B,
#         probs_C,
#         lambda yt, p: -brier_score_loss(yt, p),
#     ),
# }

# print("C (combined) vs. B (intent-only):")
#
# for metric, result in c_vs_b.items():
#     print(
#         f"  Δ{metric}: {result['delta']:+.4f} "
#         f"95% CI "
#         f"[{result['ci_lower']:+.4f}, "
#         f"{result['ci_upper']:+.4f}]"
#         f"  "
#         f"{'SIGNIFICANT' if result['significant'] else 'not significant'}"
#     )

print(
    "Question: does ASR uncertainty add anything "
    "beyond intent uncertainty on FSC?"
)

Question: does ASR uncertainty add anything beyond intent uncertainty on FSC?


## 18 — Cross-dataset comparison

The payoff cell: SLURP (Notebook 16, provisional pending its fixed-split rerun) side by
side with FSC (this notebook).

In [82]:
# Provisional SLURP Notebook 16 values — replace once its fixed-split rerun lands.
SLURP_PROVISIONAL = {
    "asr_native_roc_auc": 0.618,
    "intent_native_roc_auc": 0.911,
    "combined_roc_auc": 0.883,
}

# fsc_roc_auc = {
#     "asr_native_roc_auc": risk_results_df.query("experiment == 'A_asr_only'")["roc_auc"].max(),
#     "intent_native_roc_auc": risk_results_df.query("experiment == 'B_intent_only'")["roc_auc"].max(),
#     "combined_roc_auc": risk_results_df.query("experiment == 'C_combined'")["roc_auc"].max(),
# }
#
# cross_dataset_df = pd.DataFrame({
#     "experiment": ["ASR-native", "Intent-native", "Combined", "Combined - Intent"],
#     "SLURP": [
#         SLURP_PROVISIONAL["asr_native_roc_auc"],
#         SLURP_PROVISIONAL["intent_native_roc_auc"],
#         SLURP_PROVISIONAL["combined_roc_auc"],
#         SLURP_PROVISIONAL["combined_roc_auc"] - SLURP_PROVISIONAL["intent_native_roc_auc"],
#     ],
#     "FSC": [
#         fsc_roc_auc["asr_native_roc_auc"],
#         fsc_roc_auc["intent_native_roc_auc"],
#         fsc_roc_auc["combined_roc_auc"],
#         fsc_roc_auc["combined_roc_auc"] - fsc_roc_auc["intent_native_roc_auc"],
#     ],
# })
# cross_dataset_df.to_csv(CROSS_DATASET_PATH, index=False)
# print(cross_dataset_df.to_string(index=False))
# print("(SLURP values are provisional until Notebook 16's fixed-split rerun.)")

print("Cross-dataset table populated once Section 16 risk results are available.")

Cross-dataset table populated once Section 16 risk results are available.


In [83]:
# --- Pattern classification -------------------------------------------------
# Pattern A: SLURP C<B, FSC C>B  -> complementarity is dataset-dependent
# Pattern B: SLURP C<B, FSC C<B  -> current ASR uncertainty adds little beyond intent
# Pattern C: SLURP C>B, FSC C>B  -> strong support for H1
# Pattern D: mixed statistical evidence -> requires interpretation, not a forced conclusion

def classify_pattern(slurp_combined_beats_intent: bool, fsc_combined_beats_intent: bool,
                      fsc_ci_lower: float) -> str:
    if not slurp_combined_beats_intent and fsc_combined_beats_intent and fsc_ci_lower > 0:
        return "Pattern A — complementarity is dataset-dependent"
    if not slurp_combined_beats_intent and not fsc_combined_beats_intent:
        return "Pattern B — ASR uncertainty adds little beyond intent uncertainty"
    if slurp_combined_beats_intent and fsc_combined_beats_intent and fsc_ci_lower > 0:
        return "Pattern C — strong support for H1"
    return "Pattern D — mixed evidence, requires interpretation"

def final_verdict(combined_delta: float, ci_lower: float) -> str:
    if combined_delta > 0 and ci_lower > 0:
        return "H1 SUPPORTED"
    elif combined_delta < 0:
        return "H1 NOT SUPPORTED"
    else:
        return "H1 INCONCLUSIVE"

# verdict = final_verdict(c_vs_b["roc_auc"]["delta"], c_vs_b["roc_auc"]["ci_lower"])
# pattern = classify_pattern(
#     slurp_combined_beats_intent=SLURP_PROVISIONAL["combined_roc_auc"] > SLURP_PROVISIONAL["intent_native_roc_auc"],
#     fsc_combined_beats_intent=fsc_roc_auc["combined_roc_auc"] > fsc_roc_auc["intent_native_roc_auc"],
#     fsc_ci_lower=c_vs_b["roc_auc"]["ci_lower"],
# )
#
# print("=" * 50)
# print("NOTEBOOK 17 — CROSS-DATASET H1 VERDICT")
# print("=" * 50)
# print(f"Dataset:            Fluent Speech Commands")
# print(f"Evaluation split:   TEST")
# print(f"Examples:           {len(test_df)}")
# print()
# print(f"Intent-only ROC-AUC:  {fsc_roc_auc['intent_native_roc_auc']:.4f}")
# print(f"Combined ROC-AUC:     {fsc_roc_auc['combined_roc_auc']:.4f}")
# print(f"Delta (C - B):        {c_vs_b['roc_auc']['delta']:+.4f}")
# print(f"95% CI:               [{c_vs_b['roc_auc']['ci_lower']:+.4f}, {c_vs_b['roc_auc']['ci_upper']:+.4f}]")
# print()
# print(f"Pattern:  {pattern}")
# print(f"H1:       {verdict}")
# print("=" * 50)

print("Verdict cell defined — runs once Sections 07/09 checkpoints and Section 16/17 results exist.")

Verdict cell defined — runs once Sections 07/09 checkpoints and Section 16/17 results exist.


## Artifacts

    reports/
    ├── fsc_dataset_audit.csv
    ├── fsc_asr_predictions.csv
    ├── fsc_intent_predictions.csv
    ├── fsc_voxintel_r_features.csv
    ├── fsc_voxintel_r_model_comparison.csv
    ├── fsc_voxintel_r_predictions.csv
    ├── fsc_voxintel_r_feature_importance.csv
    ├── fsc_voxintel_r_cross_dataset_comparison.csv
    └── fsc_voxintel_r_summary.json

    models/
    ├── wav2vec2_fsc/
    └── distilbert_fsc_intent/

### Split policy (Section 30 of the design discussion)

The FSC test set is **not** used to train the risk model:

    FSC TRAIN        → ASR Transformer + Intent Transformer
    FSC VALIDATION    → checkpoint selection + risk-model development
    FSC TEST          → final, frozen VoxIntel-R evaluation only

### What happens next

Do **not** modify the finalized README's roadmap yet. Run this notebook to an actual
result first. Only then update the roadmap from

    16 → 17 calibration → 18 cost

to the experimentally justified

    16 → 17 cross-dataset validation → 18 calibration → 19 cost

so the README stays honest and the conclusion isn't written before the experiment runs.

In [84]:
summary = {
    "notebook_id": NOTEBOOK_ID,
    "dataset": DATASET,
    "hypothesis": HYPOTHESIS,
    "random_seed": RANDOM_SEED,
    "asr_features": ASR_FEATURES,
    "intent_features": INTENT_FEATURES,
    "forbidden_features_checked": sorted(FORBIDDEN_FEATURES),
    "status": "scaffold — populate after training checkpoints and running Sections 16-18",
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {SUMMARY_PATH}")

Saved: C:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\fsc_voxintel_r_summary.json
